# Oracle PLSQL Export Flow

### Exportacion manual Oracle -> Excel/CSV (1 a N sentencias)

Este notebook ejecuta sentencias SQL definidas en una variable y exporta cada resultado a un archivo independiente.

## Ruta de ejecucion
1. Configurar constantes de conexion y parametros.
2. Importar dependencias y preparar utilidades.
3. Ejecutar cada sentencia y validar columnas contra la tabla Oracle.
4. Exportar a Excel o CSV en `py_notebooks/`.
5. Revisar el reporte final consolidado.

## 1) Configuracion manual (constantes en MAYUS_SNAKE_CASE)

Edita esta celda antes de ejecutar. Define la conexion Oracle, el directorio de salida y las sentencias a exportar.

> Nota: si tu conexion usa SID, completa `ORACLE_SID`. Si usa SERVICE_NAME, completa `ORACLE_SERVICE_NAME`.

In [13]:
# ====== ORACLE ======
ORACLE_HOST = '172.19.0.9'
ORACLE_PORT = '1521'
ORACLE_SID = 'INABIF02'
ORACLE_SERVICE_NAME = None
ORACLE_USER = 'USRSEGURIDAD'
ORACLE_PWD = 'QL7nstYOMwxQ'

# ====== EXPORTACION ======
OUTPUT_DIR = './source/punche/'
DEFAULT_EXCEL_ENGINE = 'openpyxl'
EXPORT_EMPTY_RESULTS = False

# Cada sentencia genera un archivo independiente.
# table_name es obligatorio porque se usa para validar que las columnas
# del resultado coincidan exactamente con la tabla Oracle.
EXPORT_DEFINITIONS = [
   {
      'table_name': 'SSI_ANEXOS_PREGUNTAS',
      'sql': '''
            SELECT 
               -- COUNT(1) 
               ap.*
            FROM SSI_ANEXOS_PREGUNTAS ap
            WHERE
               ap.SI_ID_SERVICIO = 2
               AND ap.AP_NUM_ANEXO = 7
      ''',
      'output_format': 'excel',
      'output_name': 'ficha_identificacion.xlsx'
   }
]

## 2) Dependencias y utilidades

Si falta `oracledb`, `sqlalchemy`, `pandas` u `openpyxl`, instalalos y vuelve a ejecutar esta seccion.

In [ ]:
# %pip install pandas
# %pip install sqlalchemy
# %pip install oracledb
# %pip install openpyxl

In [14]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Dict, List

import oracledb
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger('oracle_plsql_export_flow')

def show_runtime_diagnostics() -> None:
    import os
    import sys
    print('--- Runtime Diagnostics ---')
    print(f'cwd: {os.getcwd()}')
    print(f'python: {sys.executable}')
    print(f'version: {sys.version.splitlines()[0]}')

def make_dsn(host: str, port: str, sid: str | None = None, service_name: str | None = None) -> str:
    port_num = int(port)
    if sid:
        return oracledb.makedsn(host=host, port=port_num, sid=sid)
    if service_name:
        return oracledb.makedsn(host=host, port=port_num, service_name=service_name)
    raise ValueError('Debes definir SID o SERVICE_NAME para la conexion Oracle.')

def get_connection(host: str, port: str, user: str, pwd: str, sid: str | None = None, service_name: str | None = None):
    dsn = make_dsn(host=host, port=port, sid=sid, service_name=service_name)
    return oracledb.connect(user=user, password=pwd, dsn=dsn)

def get_engine():
    dsn = make_dsn(
        host=ORACLE_HOST,
        port=ORACLE_PORT,
        sid=ORACLE_SID,
        service_name=ORACLE_SERVICE_NAME
    )
    connection_string = f'oracle+oracledb://{ORACLE_USER}:{ORACLE_PWD}@{dsn}'
    return create_engine(connection_string, poolclass=NullPool, echo=False)

def get_table_columns(conn, owner: str, table_name: str) -> List[str]:
    sql = (
        'SELECT COLUMN_NAME FROM ALL_TAB_COLUMNS ' 
        'WHERE OWNER = :owner AND TABLE_NAME = :table_name ' 
        'ORDER BY COLUMN_ID'
    )
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper(), table_name=table_name.upper())
        rows = cur.fetchall()
    if not rows:
        raise ValueError(f'No se encontro metadata para {owner}.{table_name}')
    return [row[0] for row in rows]

def execute_query(query: str, params: Dict | None = None) -> pd.DataFrame:
    engine = None
    try:
        engine = get_engine()
        with engine.connect() as conn:
            if params:
                df = pd.read_sql_query(sql=text(query), con=conn, params=params)
            else:
                df = pd.read_sql_query(sql=text(query), con=conn)
        logger.info('Consulta ejecutada exitosamente, filas encontradas: %s', len(df))
        return df
    finally:
        if engine is not None:
            engine.dispose()

def normalize_export_definitions(definitions: List[Dict]) -> List[Dict]:
    normalized = []
    valid_formats = {'excel', 'csv'}
    for index, item in enumerate(definitions, start=1):
        if not isinstance(item, dict):
            raise ValueError(f'EXPORT_DEFINITIONS[{index}] debe ser dict.')

        table_name = str(item.get('table_name', '')).strip().upper()
        sql = str(item.get('sql', '')).strip()
        output_format = str(item.get('output_format', '')).strip().lower()
        output_name = str(item.get('output_name', '')).strip()

        if not table_name:
            raise ValueError(f'EXPORT_DEFINITIONS[{index}] requiere table_name.')
        if not sql:
            raise ValueError(f'EXPORT_DEFINITIONS[{index}] requiere sql.')
        if output_format not in valid_formats:
            raise ValueError(f'EXPORT_DEFINITIONS[{index}] tiene output_format invalido: {output_format}')
        if not output_name:
            raise ValueError(f'EXPORT_DEFINITIONS[{index}] requiere output_name.')

        normalized.append({
            'table_name': table_name,
            'sql': sql,
            'output_format': output_format,
            'output_name': output_name
        })

    if not normalized:
        raise ValueError('Debes definir al menos una sentencia en EXPORT_DEFINITIONS.')
    return normalized

def validate_export_columns(df: pd.DataFrame, expected_columns: List[str], table_name: str) -> None:
    result_columns = [str(col).upper() for col in df.columns.tolist()]
    expected = [str(col).upper() for col in expected_columns]
    if result_columns != expected:
        raise ValueError(
            'Las columnas del resultado no coinciden exactamente con la tabla ' 
            f'{table_name}. Esperado={expected}; obtenido={result_columns}'
        )

def build_output_path(output_name: str) -> Path:
    output_dir = Path(OUTPUT_DIR).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / output_name

def export_dataframe(df: pd.DataFrame, output_path: Path, output_format: str) -> None:
    if output_format == 'excel':
        df.to_excel(output_path, index=False, engine=DEFAULT_EXCEL_ENGINE)
        return
    if output_format == 'csv':
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        return
    raise ValueError(f'Formato no soportado: {output_format}')

def run_export_definition(conn, export_definition: Dict, owner: str) -> Dict:
    table_name = export_definition['table_name']
    output_format = export_definition['output_format']
    output_name = export_definition['output_name']

    result = {
        'table_name': table_name,
        'output_format': output_format,
        'output_file': '',
        'rows_exported': 0,
        'status': 'OK',
        'message': ''
    }

    try:
        expected_columns = get_table_columns(conn, owner, table_name)
        df = execute_query(export_definition['sql'])

        if df.empty and not EXPORT_EMPTY_RESULTS:
            raise ValueError(f'La sentencia para {table_name} no retorno filas.')

        validate_export_columns(df, expected_columns, table_name)

        output_path = build_output_path(output_name)
        export_dataframe(df, output_path, output_format)

        result['output_file'] = str(output_path)
        result['rows_exported'] = len(df)
        result['message'] = 'Exportacion completada.'
        logger.info('Archivo generado: %s', output_path)
    except Exception as ex:
        result['status'] = 'ERROR'
        result['message'] = str(ex)
        logger.exception('Error exportando definicion para tabla %s', table_name)

    return result

## 2.1) Diagnostico rapido del kernel

Ejecuta esta celda si tienes errores de import o dudas sobre el entorno activo de Jupyter.

In [ ]:
show_runtime_diagnostics()

## 3) Ejecucion manual (paso principal)

Se procesa cada sentencia definida en `EXPORT_DEFINITIONS`. La exportacion solo ocurre si las columnas del resultado coinciden exactamente con la tabla Oracle configurada.

In [15]:
results = []
conn = None

try:
    export_definitions = normalize_export_definitions(EXPORT_DEFINITIONS)
    owner = ORACLE_USER.upper()

    conn = get_connection(
        host=ORACLE_HOST,
        port=ORACLE_PORT,
        user=ORACLE_USER,
        pwd=ORACLE_PWD,
        sid=ORACLE_SID,
        service_name=ORACLE_SERVICE_NAME
    )

    logger.info('Total sentencias a procesar: %s', len(export_definitions))

    for export_definition in export_definitions:
        logger.info('Procesando tabla objetivo: %s', export_definition['table_name'])
        results.append(run_export_definition(conn, export_definition, owner))
finally:
    if conn is not None:
        conn.close()

2026-06-03 19:31:59,705 | INFO | Total sentencias a procesar: 1
2026-06-03 19:31:59,706 | INFO | Procesando tabla objetivo: SSI_ANEXOS_PREGUNTAS
2026-06-03 19:31:59,765 | INFO | Consulta ejecutada exitosamente, filas encontradas: 44
2026-06-03 19:31:59,781 | INFO | Archivo generado: C:\work-space\sigeif-app\sigeif_ssi_inabif_plsql\py_notebooks\source\punche\ficha_identificacion.xlsx


## 4) Reporte final y lectura rapida

Revisa el estado por sentencia, archivo generado, formato y total de filas exportadas.

In [12]:
print('=' * 60)
print('ORACLE'.center(60))
print('=' * 60)
print(f'  Host   : {ORACLE_HOST}:{ORACLE_PORT}')
print(f'  User   : {ORACLE_USER}')
print(f'  SID    : {ORACLE_SID or ORACLE_SERVICE_NAME}')
print(f'  Output : {Path(OUTPUT_DIR).expanduser().resolve()}')
print()
print(f'Total resultados: {len(results)}')
print()

if not results:
    print('No hay resultados para mostrar.')
else:
    try:
        report_df = pd.DataFrame(results)
        try:
            from IPython.display import display
            display(report_df)
        except Exception:
            print(report_df.to_string(index=False))
    except Exception as ex:
        print(f'Error al generar DataFrame: {ex}')
        for item in results:
            print(item)

                           ORACLE                           
  Host   : 172.19.0.9:1521
  User   : USRSEGURIDAD
  SID    : INABIF02
  Output : C:\work-space\sigeif-app\sigeif_ssi_inabif_plsql\py_notebooks\py_notebooks\source\punche

Total resultados: 1



,table_name,output_format,output_file,rows_exported,status,message
0,SSI_ANEXOS_PREGUNTAS,excel,C:\work-space\sigeif-app\sigeif_ssi_inabif_pls...,44,OK,Exportacion completada.
